In [2]:
import numpy as np
import math

from functools import reduce
from itertools import product

Brute force worked pretty well right away. I did some proving to show that no solutions exist with $5$ digits or with $9$ digits, which sped up the process significantly.

In [38]:
##############################
# First Brute Force Attempt
##############################

sols = []
for places in [6]:
    # we know no solutions with 5 digits or 9 digits
    if places in [5,9]: continue

    seen = set()
    for tup in product(range(10), repeat=places):
        if tup in seen: continue
        reverse_tup = tup[::-1]
        seen.add(tup)
        seen.add(reverse_tup)

        # no leading or trailing zeros
        if tup[0] == 0 or tup[-1] == 0: continue

        # first and last digits need to be opposite parity
        if (tup[0] + tup[-1]) % 2 == 0: continue

        # for places == 7, we need first and last digit to be >10
        if places == 7:
            if (tup[0] + tup[6]) <= 10: continue
            if (tup[1] + tup[5]) % 2 == 1: continue
            if (tup[2] + tup[4]) <= 10: continue

        # # check if opposite pairs of numbers are opposite parity
        # for i in range(places//2):
        #     if (tup[i] - tup[places-i-1]) % 2 == 0:
        #         continue

        n = sum([(tup[i]) * (10**i) for i in range(places)])
        reverse_n = sum([(reverse_tup[i]) * (10**i) for i in range(places)])

        to_check = tuple([int(d) for d in str(n+reverse_n)])

        if reduce(lambda a, b: (a & b), to_check, 1):
            print(tup, reverse_tup, n, reverse_n, n + reverse_n)
            sols.append(n)
            sols.append(reverse_n)

len(sols)

(1, 0, 0, 1, 1, 2) (2, 1, 1, 0, 0, 1) 211001 100112 311113
(1, 0, 0, 1, 1, 4) (4, 1, 1, 0, 0, 1) 411001 100114 511115
(1, 0, 0, 1, 1, 6) (6, 1, 1, 0, 0, 1) 611001 100116 711117
(1, 0, 0, 1, 1, 8) (8, 1, 1, 0, 0, 1) 811001 100118 911119
(1, 0, 0, 1, 3, 2) (2, 3, 1, 0, 0, 1) 231001 100132 331133
(1, 0, 0, 1, 3, 4) (4, 3, 1, 0, 0, 1) 431001 100134 531135
(1, 0, 0, 1, 3, 6) (6, 3, 1, 0, 0, 1) 631001 100136 731137
(1, 0, 0, 1, 3, 8) (8, 3, 1, 0, 0, 1) 831001 100138 931139
(1, 0, 0, 1, 5, 2) (2, 5, 1, 0, 0, 1) 251001 100152 351153
(1, 0, 0, 1, 5, 4) (4, 5, 1, 0, 0, 1) 451001 100154 551155
(1, 0, 0, 1, 5, 6) (6, 5, 1, 0, 0, 1) 651001 100156 751157
(1, 0, 0, 1, 5, 8) (8, 5, 1, 0, 0, 1) 851001 100158 951159
(1, 0, 0, 1, 7, 2) (2, 7, 1, 0, 0, 1) 271001 100172 371173
(1, 0, 0, 1, 7, 4) (4, 7, 1, 0, 0, 1) 471001 100174 571175
(1, 0, 0, 1, 7, 6) (6, 7, 1, 0, 0, 1) 671001 100176 771177
(1, 0, 0, 1, 7, 8) (8, 7, 1, 0, 0, 1) 871001 100178 971179
(1, 0, 0, 1, 9, 2) (2, 9, 1, 0, 0, 1) 291001 100192 3911

18000

In [61]:
#################################
# Second brute force attempt
#################################

cnt = 0
for places in [2,3,4,5,6,7,8]:
    # we know no solutions with 5 digits or 9 digits
    if places in [5,9]: continue
    
    seen = set()

    # if even number of places each layer is symmetric but if odd then middle layer can be anything
    odd_layer = [1,3,5,7,9]
    middle_layer = [0,1,2,3,4,5,6,7,8,9]
    even_layer = [0,2,4,6,8]
    
    if places % 2:
        layers = []
        layers.append(even_layer)
        layers += [middle_layer for _ in range(1, places-1)]
        layers.append(odd_layer)

        for tup in product(*layers):
            if tup in seen: continue
            reverse_tup = tup[::-1]
            seen.add(tup)
            seen.add(reverse_tup)

            # no leading or trailing zeros
            if tup[0] == 0 or tup[-1] == 0: continue

            # first and last digits need to be opposite parity
            if (tup[0] + tup[-1]) % 2 == 0: continue

            # for places is odd, we need even layers to be > 10 and odd layers to be same parity
            even = True
            for i in range(places//2):
                if even:
                    if tup[i] + reverse_tup[i] <= 10:
                        continue

                    even = False
                else:
                    if (tup[i] + reverse_tup[i]) % 2 == 1:
                        continue

                    even = True

            to_check = 0
            for i in range(places): to_check += (tup[i] + reverse_tup[i])*10**i
            to_check = [((to_check) % 10**(i+1))//10**i for i in range(places)]

            if reduce(lambda a, b: (a & b), to_check, 1):
                # print(tup, reverse_tup, to_check)
                # sols.append(n)
                # sols.append(reverse_n)
                cnt += 2

    else:
        lc_seen = set()
        for layer_choices in product([0,1], repeat=places//2):
            if layer_choices in lc_seen: continue
            lc_seen.add(layer_choices)
            lc_seen.add(tuple([1 - lc for lc in layer_choices]))

            layers = [[] for _ in range(places)]

            for i, lc in enumerate(layer_choices):
                if lc:
                    layers[i] = odd_layer
                    layers[places-i-1] = even_layer
                else:
                    layers[i] = even_layer
                    layers[places-i-1] = odd_layer
        
            for tup in product(*layers):
                if tup in seen: continue
                reverse_tup = tup[::-1]
                seen.add(tup)
                seen.add(reverse_tup)

                # no leading or trailing zeros
                if tup[0] == 0 or tup[-1] == 0: continue

                # first and last digits need to be opposite parity
                if (tup[0] + tup[-1]) % 2 == 0: continue

                to_check = 0
                for i in range(places): to_check += (tup[i] + reverse_tup[i])*10**i
                to_check = [((to_check) % 10**(i+1))//10**i for i in range(places)]

                if reduce(lambda a, b: (a & b), to_check, 1):
                    # print(tup, reverse_tup, to_check)
                    # sols.append(n)
                    # sols.append(reverse_n)
                    cnt += 2

cnt

608720